### Mini Project IX

#### Part 2 — Baseline (TF-IDF + Logistic Regression)

Train and evaluate a simple tf-idf + logistic regression model as a baseline for the content moderation task.

In [ ]:
# Imports
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import matplotlib.pyplot as plt
import seaborn as sns

# Seed for reproducibility
SEED = 42
np.random.seed(SEED)

print("✅ Imports loaded")

In [ ]:
# Load preprocessed data splits (created in Part 1)
CLASS_NAMES = {0: "Hate Speech", 1: "Offensive", 2: "Neither"}
LABELS = list(CLASS_NAMES.values())

X_train = np.load("X_train.npy", allow_pickle=True)
X_val   = np.load("X_val.npy",   allow_pickle=True)
X_test  = np.load("X_test.npy",  allow_pickle=True)

y_train = np.load("y_train.npy", allow_pickle=True)
y_val   = np.load("y_val.npy",   allow_pickle=True)
y_test  = np.load("y_test.npy",  allow_pickle=True)

# Combine train+val so we can refit the best model after tuning
X_trainval = np.concatenate([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

# Build pipeline: TF-IDF -> Logistic Regression
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        sublinear_tf=True,   # apply log normalization to term freq
        min_df=3,            # ignore very rare terms
        token_pattern=r"\b\w\w+\b"
    )),
    ("clf", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=SEED,
        solver="lbfgs",
        multi_class="multinomial"
    ))
])

# Hyperparameter grid for tuning
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [20_000, 50_000],
    "clf__C": [0.1, 1.0, 10.0]
}

# Grid search over tune set (train split)
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print("Best params :", grid_search.best_params_)
print(f"Best CV macro-F1: {grid_search.best_score_:.4f}")

# Validate on held-out validation set
best_model = grid_search.best_estimator_
y_val_pred = best_model.predict(X_val)
print(f"\nVal accuracy : {accuracy_score(y_val, y_val_pred):.4f}")
print("\nVal classification report:")
print(classification_report(y_val, y_val_pred, target_names=LABELS))

# Train final model on train+val and evaluate on test
pipeline.set_params(**grid_search.best_params_)
pipeline.fit(X_trainval, y_trainval)
y_test_pred = pipeline.predict(X_test)

print(f"Test accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
print("\nTest classification report:")
print(classification_report(y_test, y_test_pred, target_names=LABELS))

# Save metrics for later comparison
report_dict = classification_report(
    y_test, y_test_pred, target_names=LABELS, output_dict=True
)
metrics_df = pd.DataFrame(report_dict).T
metrics_df.to_csv("baseline_metrics.csv")

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("TF-IDF + Logistic Regression — Test Set", fontsize=13, fontweight="bold")

cm = confusion_matrix(y_test, y_test_pred)
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(
    ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title("Confusion Matrix (counts)")
axes[0].set_xticklabels(LABELS, rotation=15, ha="right")

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS,
            ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("Confusion Matrix (row-normalised / recall)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")
axes[1].set_xticklabels(LABELS, rotation=15, ha="right")

plt.tight_layout()
plt.savefig("baseline_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

# Print baseline summary
print("\nBaseline summary (test set):")
for cls in LABELS + ["macro avg"]:
    r = report_dict[cls]
    print(f"  {cls:<18} P={r['precision']:.3f}  R={r['recall']:.3f}  F1={r['f1-score']:.3f}")
